# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

- **Gymnasium es el ambiente (`env`).** Implementa la dinámica del mundo: `reset()` entrega el estado inicial y `step(action)` devuelve `(next_state, reward, terminated, truncated, info)`. Cumple el papel de la clase `Environment` de CheeseWorld.
- **El agente no es una clase:** está repartido en la `Qtable` (su conocimiento) y las funciones `epsilon_greedy_policy`, `greedy_policy` y `train_q_learning` (cómo decide y cómo actualiza). En CheeseWorld eso estaba dentro de `QLearningAgent`.
- Gymnasium solo ve acciones: no entrega las probabilidades de transición. El agente aprende solo de la experiencia `(s, a, r, s')`.

## 2. Crear FrozenLake


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

- **Filas:** 16, una por estado (`observation_space.n`, la cuadrícula 4×4).
- **Columnas:** 4, una por acción (izquierda, abajo, derecha, arriba).
- **`Q[s,a]`:** el retorno descontado esperado si en el estado `s` se ejecuta `a` y después se sigue la mejor política. No es solo la recompensa inmediata; incluye lo que se espera ganar a futuro.

In [3]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- Que todos los valores sean 0 significa que **el agente todavía no sabe nada**, no que todas las acciones sean malas. Es una inicialización sin información.
- Si varias acciones empatan en el máximo, `argmax` escogería siempre la primera (sesgo hacia una dirección). Por eso `greedy_policy` desempata **al azar** entre las acciones con el mismo Q.
- Al inicio, la política greedy se comporta como una caminata aleatoria hasta que las recompensas empiezan a diferenciar valores.

## 3. Ejecutar una transición


In [4]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 3
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(s,a,r,s')=(0,\;3,\;0,\;0)
$$

Estado 0, acción 3 (Up), recompensa 0 y siguiente estado 0: estando en la esquina superior izquierda, ir hacia arriba choca con el borde y el agente se queda en el mismo estado.

**Pregunta:** en Value Iteration usábamos el modelo completo `T(s,a,s')` y `R(s)`, es decir, **conocíamos de antemano** la probabilidad de cada siguiente estado. Con experiencia real no disponemos de `T(s,a,s')`: solo observamos **una muestra** `s'` (y `r`) cada vez que actuamos.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [5]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [6]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [7]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [8]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.

**Respuesta:** el ambiente, los estados y las acciones no cambian; lo único que cambió es la **Q-table**. En `Q_initial` todo vale 0 y el agente actúa al azar. En `Q_early` (50 episodios) solo tienen valor las acciones cercanas a donde se obtuvo alguna recompensa, así que la política es parcial y puede fallar. En `Q_trained` (10 000 episodios) la recompensa de la meta ya se propagó hacia atrás, mediante la actualización TD, a todos los estados del camino, por lo que `argmax_a Q(s,a)` indica en cada estado la acción que lleva a la meta (recompensa 1.0 frente a 0.0 de las otras dos ejecuciones).

In [ ]:
state = 14
print("Estado:", state)
print("Q-values [Left, Down, Right, Up]:", Q_trained[state].round(3))
print("Mejor acción:", int(np.argmax(Q_trained[state])))

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** 14 (fila 3, columna 2: justo a la izquierda de la meta)

**Valores Q** (valores a los que converge Q-Learning con `gamma = 0.95` en este mapa determinístico; la celda de código de arriba imprime los de tu `Q_trained`, que deben ser muy parecidos):

- Left: ≈ 0.90
- Down: ≈ 0.95 (choca con el borde y se queda en el 14)
- Right: ≈ 1.00 (llega a la meta, recompensa 1)
- Up: ≈ 0.90

La acción `argmax_a Q(s,a)` es **Right (2)**.

**Interpretación:**

> Right vale 1.00 porque produce la recompensa directa. Las demás acciones valen algo menos, pero no cero, porque desde el siguiente estado todavía se puede llegar a la meta: ese valor es `gamma` multiplicado por el valor del mejor camino restante. Por eso Q no es la recompensa inmediata (que es 0 en todas ellas salvo Right), sino un retorno descontado.

## 5. Evaluar la política aprendida


In [9]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> Exploramos con ε-greedy durante el entrenamiento para descubrir recompensas y no quedarnos con una política subóptima. En la evaluación queremos **medir la calidad de la política aprendida**, no seguir aprendiendo: si siguiéramos tomando acciones aleatorias con probabilidad ε, mediríamos una mezcla de la política y del azar, y obtendríamos una recompensa menor y más ruidosa. Por eso se usa la política greedy `a = argmax_a Q(s,a)` (explotación pura). Aquí la evaluación dio media 1.0 y desviación 0.0: el agente llega a la meta en los 100 episodios.

## 6. Experimento: FrozenLake estocástico


In [10]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.000
Std reward : 0.000


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- **Ambiente:** con `is_slippery=False` la acción produce siempre el movimiento elegido (`T(s,a,s')=1` para un único `s'`). Con `is_slippery=True` el agente se desliza: solo en 1/3 de los casos va a la dirección elegida y en los otros 2/3 se mueve en una dirección perpendicular. La transición pasa a ser una distribución de probabilidad y la misma experiencia `(s,a)` puede dar distintos `s'`.
- **Algoritmo:** la ecuación de Q-Learning **no cambia**: `Q(s,a) ← Q(s,a) + α [r + γ max_a' Q(s',a') − Q(s,a)]`. Como el algoritmo es model-free, la estocasticidad solo se refleja en que el objetivo TD es ruidoso: el promedio de muchas muestras aproxima la esperanza sobre `s'`. Lo que cambia es la **dificultad**: se necesitan más episodios y una tasa de aprendizaje menor.
- En el notebook, el agente resbaladizo se evaluó con media 0.000. Con `learning_rate=0.7` cada muestra ruidosa sobrescribe casi por completo el valor anterior, así que la Q-table oscila y no se estabiliza. Sería razonable repetirlo con `learning_rate` ≈ 0.05–0.1 y más episodios.

# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> El retorno descontado esperado de tomar la acción `a` en el estado `s` y seguir después la mejor política: recompensa inmediata más lo que se espera obtener a futuro.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> De la propia Q-table, en la fila del siguiente estado observado `s'`. Es la estimación actual del valor del mejor camino desde `s'` (el *bootstrap*): se actualiza una estimación usando otra estimación. En estados terminales vale 0.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar exploración y explotación. Si solo se explota, el agente se queda con las primeras acciones que dieron recompensa y nunca descubre mejores caminos; si solo se explora, nunca aprovecha lo aprendido. Con ε alto al inicio y decreciente, explora primero y luego explota.

**4. ¿Por qué Q-Learning es model-free?**

> Porque no necesita conocer ni estimar la función de transición $T(s,a,s')$ ni la de recompensa $R$. Aprende solo de experiencias $(s,a,r,s')$ obtenidas interactuando con el ambiente, a diferencia de Value Iteration y Policy Iteration, que usan el modelo explícito.